# DP2 tract 9813: metadetect shear → Kaiser–Squires mass map

Contact author: Alex Broughton  <br>Date: April 14, 2026


In [2]:
! eups list -s | grep lsst_distrib

/bin/bash: eups: command not found


## Introduction

This notebook builds a **single-tract** weak-lensing convergence map (**κ**) for **tract 9813** from DP2 **metadetect** catalogs via the Rubin Butler. Sources are binned onto **HEALPix** pixels at `CONFIG["HEALPIX_NSIDE"]` (nested ordering, Rubin-style), with pixel size set by NSIDE (e.g. **512 → ~7′**, **1024 → ~3.4′**—cluster-scale). **Kaiser–Squires** is applied in **curved-sky** form via spherical harmonics (`kaiser_squires_healpix`), so no tangent-pointing or tangent-plane center is required for binning or inversion.

**References**

- Butler access and dataset overview: [`_tutorials/access-dp2-metadetect-catalogs.ipynb`](../_tutorials/access-dp2-metadetect-catalogs.ipynb)
- Selection ideas (re-expressed here for DP2 columns): [lsst-so/comcam_clusters — `A360_shear_profile_metadetect.ipynb`](https://github.com/lsst-so/comcam_clusters/blob/main/A360_shear_profile_metadetect.ipynb), [`ACO360_SourceSelection.ipynb`](https://github.com/lsst-so/comcam_clusters/blob/main/ACO360_SourceSelection.ipynb)
- Mass mapping on small cutouts in that repo uses aperture statistics; this notebook uses **curved-sky KS** for tract-scale maps instead of the `ACO360_WL_HSCcalib_MassMap.ipynb` double loop. **Do not** use the ComCam **HSC shear calibration** path: DP2 metadetect uses the **noshear** branch (schema column **`ns`**, or `shear_type` / `mdet_step` when present).

**Caveats**

- **Partial-sky / sparse HEALPix:** spherical-harmonic KS has mode-mixing and boundary effects on a tract-sized footprint unless you use masking-aware or iterative methods; treat κ_B as a diagnostic only.
- **Memory:** lowering `HEALPIX_NSIDE` reduces map size (`12 NSIDE²` floats per field); a full tract catalog can be large—trim patches or rows for tests.
- Column names are **inferred** from the Arrow schema on first load; verify on SDF if inference fails.

**Future (not implemented here):** full-DP2 runs as parallel tract jobs, peak finding on saved κ arrays, and **N** shear-angle nulls with `numpy.random.Generator` seeded from `CONFIG["RANDOM_SEED"]` + job id—see markdown section 8.0.


## 1.0 Set up

`CONFIG` holds everything a future cluster script would mirror with CLI args. Core logic lives in importable pure functions in [`lib/tract_massmap.py`](lib/tract_massmap.py).


In [3]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import pyarrow as pa

from lsst.daf.butler import Butler
from lsst.skymap import BaseSkyMap

# Repo root: cwd is repo or notebooks/
for base in (Path.cwd(), Path.cwd().parent):
    if (base / "notebooks" / "lib" / "tract_massmap.py").is_file():
        if str(base / "notebooks") not in sys.path:
            sys.path.insert(0, str(base / "notebooks"))
        REPO_ROOT = base
        break
else:
    raise RuntimeError("Run from the git repo root or from notebooks/.")

from lib.tract_massmap import (
    ShearColumnMap,
    bin_shear_healpix,
    bin_shear_weighted,
    columns_for_loading,
    filter_noshear_table,
    kaiser_squires,
    kaiser_squires_healpix,
    list_patches_for_tract,
    load_patches_concat,
    select_sources_basic,
)

CONFIG = {
    "REPO": "/sdf/data/rubin/repo/dp2_prep",
    "COLLECTION": "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "SKYMAP": "lsst_cells_v2",
    "INSTRUMENT": "LSSTCam",
    "TRACT": 9813,
    # HEALPix (nested): 512→~6.9′, 1024→~3.4′, 2048→~1.7′ per pixel
    "HEALPIX_NSIDE": 512,
    # Grid / tangent-plane FFT path (optional; bin_shear_weighted)
    "GRID_NX": 512,
    "GRID_NY": 512,
    "PIXEL_SCALE_ARCSEC": 0.2,
    # Selection (tune for science)
    "MIN_WEIGHT": 0.0,
    "MAX_ABS_G": 1.0,
    # Debug: set to an int to load only the first N patches with data
    "MAX_PATCHES": None,
    # Reproducibility placeholder for future angle-null jobs (not used in v1)
    "RANDOM_SEED": 42,
}

%matplotlib inline
plt.rcParams.update({"font.size": 12})

print("REPO_ROOT =", REPO_ROOT)
print("TRACT =", CONFIG["TRACT"])


ModuleNotFoundError: No module named 'lsst'

## 2.0 Butler and patch list

Query the registry for every `object_shear_patch` dataset in this tract (do not load `object_shear_all` whole-survey tables into memory).


In [ ]:
butler = Butler(
    CONFIG["REPO"],
    collections=[CONFIG["COLLECTION"]],
    skymap=CONFIG["SKYMAP"],
    instrument=CONFIG["INSTRUMENT"],
)

patches = list_patches_for_tract(butler, CONFIG["TRACT"], CONFIG["COLLECTION"])
print(f"Found {len(patches)} patches with object_shear_patch for tract {CONFIG['TRACT']}")


Found 100 patches with object_shear_patch for tract 9813


## 3.0 Load catalogs

Load one patch to infer column names, then reload a minimal column subset for all patches (faster I/O, lower memory).


In [3]:
data = butler.get("object_shear_all", dataId={"tract": CONFIG["TRACT"]})

In [ ]:
# gauss_g1
# gauss_g2
# gauss_g1_g1_Cov
# gauss_g1_g2_Cov
# gauss_g2_g2_Cov

## 4.0 Source selection

Keep **noshear** metadetect rows only, then apply conservative cuts in `select_sources_basic` (see `tract_massmap.py`).


In [1]:
mask = (data.columns['metaStep'] == "ns")
mask &= (data['image_flags'] == 0)
mask &= (data['psfOriginal_flags'] == 0)
mask &= (data['bmask_flags'] == 0)
mask &= (data['ormask_flags'] == 0)
mask &= (data['mfrac'] < 0.1)

# Mag cut?

shear_catalog = data[mask]
print("Sources before quality cuts:", len(data))
print("Sources after quality cuts:", len(shear_catalog))


NameError: name 'data' is not defined

## 5.0 HEALPix weighted binning


In [ ]:
g1_hpx, g2_hpx, w_hpx, hits_hpx, nside = bin_shear_healpix(
    shear_catalog,
    nside=CONFIG["HEALPIX_NSIDE"],
)
npix = 12 * nside**2
print(
    f"NSIDE={nside}, npix={npix}, "
    f"mean pixel ~ {hp.nside2resol(nside, arcmin=True):.2f} arcmin"
)
print("Hit pixels:", int((hits_hpx > 0).sum()), "/", npix)


## 6.0 Kaiser–Squires inversion

**Curved sky:** `map2alm_spin` + harmonic KS multiplier (no FFT pixel scale). For a small patch you can still use `kaiser_squires` on a tangent-plane grid with `bin_shear_weighted`.


In [ ]:
kappa_e, kappa_b = kaiser_squires_healpix(g1_hpx, g2_hpx, nside=nside)
print("kappa_E range:", float(np.nanmin(kappa_e)), float(np.nanmax(kappa_e)))
print("kappa_B std (noise-like if small systematics):", float(np.std(kappa_b)))


## 7.0 Figures

E-mode κ (lensing signal) and B-mode map (approximate null).


In [ ]:
ra_plot = shear_catalog["ra"].combine_chunks().to_numpy(zero_copy_only=False)
dec_plot = shear_catalog["dec"].combine_chunks().to_numpy(zero_copy_only=False)
pad = 1.5
lonra = [float(np.mean(ra_plot)) - pad, float(np.mean(ra_plot)) + pad]
latra = [float(np.mean(dec_plot)) - pad, float(np.mean(dec_plot)) + pad]

msk = hits_hpx > 0
if msk.any():
    k_e_lo, k_e_hi = np.nanpercentile(kappa_e[msk], [5, 95])
    k_b_lo, k_b_hi = np.nanpercentile(kappa_b[msk], [5, 95])
else:
    k_e_lo = k_e_hi = k_b_lo = k_b_hi = None

fig = plt.figure(figsize=(14, 4.2))
hp.cartview(
    np.log10(hits_hpx + 0.1),
    fig=1,
    sub=(1, 3, 1),
    nest=True,
    title=r"log10(hit count + 0.1)",
    cmap="magma",
    lonra=lonra,
    latra=latra,
)
hp.cartview(
    kappa_e,
    fig=1,
    sub=(1, 3, 2),
    nest=True,
    title=r"$\kappa_E$ (Kaiser–Squires, curved sky)",
    cmap="RdBu_r",
    min=k_e_lo,
    max=k_e_hi,
    lonra=lonra,
    latra=latra,
)
hp.cartview(
    kappa_b,
    fig=1,
    sub=(1, 3, 3),
    nest=True,
    title=r"$\kappa_B$ (null)",
    cmap="RdBu_r",
    min=k_b_lo,
    max=k_b_hi,
    lonra=lonra,
    latra=latra,
)
plt.suptitle(f"Tract {CONFIG['TRACT']} — DP2 metadetect noshear (NSIDE={nside})", y=1.02)
plt.tight_layout()

fig2, ax = plt.subplots(figsize=(6, 4))
ax.hist(kappa_b[msk], bins=80, color="steelblue", alpha=0.85)
ax.set_xlabel(r"$\kappa_B$")
ax.set_ylabel("HEALPix pixels with sources")
ax.set_title(r"$\kappa_B$ histogram (hit pixels only)")
fig2.tight_layout()


## 8.0 Future: scale, peaks, nulls, scripts

**Full DP2 footprint:** run **one job per tract** (or per tile), write `npz` / FITS / Zarr with κ, γ grids, and WCS for downstream stacking and correlation with survey-property maps.

**Peaks:** treat κ and optional κ_B as numpy arrays + `astropy.wcs.WCS` so a separate peak-finder step can consume saved maps.

**Shear-angle nulls:** for realization `i`, rotate `g1 + i g2` by a fixed or random angle with `numpy.random.Generator(CONFIG["RANDOM_SEED"] + i)`, re-bin, re-run KS; submit **N** independent cluster jobs.

**Notebook → script:** mirror `CONFIG` with `argparse` or YAML, `import lib.tract_massmap`, log Butler **collection**, **skymap**, tract list, cut thresholds, grid shape, and git hash / `eups` versions.
